# Crack Segmentation — MAnet + mit_b2 (False Positive Fix)
Original training + FP mitigation: threshold tuning + post-processing

In [ ]:
import torch
assert torch.cuda.is_available()
print(f'GPU: {torch.cuda.get_device_name(0)}')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
from pathlib import Path
import json, zipfile, shutil, cv2, numpy as np, random
from sklearn.model_selection import train_test_split

WORK_DIR = Path('/content')
DATA_DIR = WORK_DIR / 'data'
CKPT_DIR = WORK_DIR / 'checkpoints'
DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')

def extract_dataset(zip_name):
    zip_path = DRIVE_DIR / zip_name
    if not zip_path.exists():
        return
    out_dir = DATA_DIR / zip_name.replace('.zip', '')
    if out_dir.exists() and any(out_dir.rglob('*.*')):
        return
    print(f'Extracting {zip_name}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)

for z in ['masonry.zip', 'crackforest.zip', 'historical_crack.zip']:
    extract_dataset(z)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
IMG_DIR_NAMES = {'images', 'img', 'image'}
MASK_DIR_NAMES = {'masks', 'mask', 'labels', 'label', 'annotations'}

def find_pairs(root_dir):
    pairs, seen = [], set()
    for img_dir in Path(root_dir).rglob('*'):
        if not img_dir.is_dir() or img_dir.name.lower() not in IMG_DIR_NAMES:
            continue
        mask_dir = None
        for mn in MASK_DIR_NAMES:
            if (img_dir.parent / mn).is_dir():
                mask_dir = img_dir.parent / mn
                break
        if not mask_dir:
            continue
        for img in sorted(img_dir.glob('*.*')):
            if img.suffix.lower() not in IMG_EXTS or str(img) in seen:
                continue
            for ext in ['.png', '.jpg', img.suffix]:
                msk = mask_dir / f'{img.stem}{ext}'
                if msk.exists():
                    pairs.append((img, msk))
                    seen.add(str(img))
                    break
    return pairs

all_pairs = []
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir():
        p = find_pairs(d)
        all_pairs.extend(p)
        print(f'{d.name}: {len(p)} pairs')

print(f'Total: {len(all_pairs)} pairs')
train_p, temp_p = train_test_split(all_pairs, train_size=0.7, random_state=42)
val_p, test_p = train_test_split(temp_p, train_size=0.5, random_state=42)
print(f'Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}')

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
_CLAHE = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def get_transforms(split, size=384):
    if split == 'train':
        return A.Compose([
            A.Resize(size, size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(p=0.1),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    return A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

class SegDataset(Dataset):
    def __init__(self, pairs, split='train', size=384):
        self.pairs = pairs
        self.transform = get_transforms(split, size)
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img_p, mask_p = self.pairs[idx]
        img = cv2.imread(str(img_p))
        img = np.zeros((512, 512, 3), dtype=np.uint8) if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[..., 0] = _CLAHE.apply(lab[..., 0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
        mask = np.zeros((512, 512), dtype=np.uint8) if mask is None else mask
        if mask.shape[:2] != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        aug = self.transform(image=img, mask=mask)
        return aug['image'], aug['mask'].unsqueeze(0)

def make_loaders(train_pairs, val_pairs, test_pairs, size, batch):
    return (
        DataLoader(SegDataset(train_pairs, 'train', size), batch_size=batch, shuffle=True, num_workers=2),
        DataLoader(SegDataset(val_pairs, 'val', size), batch_size=4, shuffle=False, num_workers=2),
        DataLoader(SegDataset(test_pairs, 'test', size), batch_size=4, shuffle=False, num_workers=2),
    )

In [ ]:
import torch
import segmentation_models_pytorch as smp
from torch.amp import GradScaler, autocast
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda')
model = smp.MAnet(encoder_name='mit_b2', encoder_weights='imagenet', in_channels=3, classes=1, activation=None).to(DEVICE)
print(f'Model: MAnet+mit_b2  Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

tversky_loss = smp.losses.TverskyLoss(mode='binary', alpha=0.3, beta=0.7, from_logits=True)
lovasz_loss = smp.losses.LovaszLoss(mode='binary', per_image=False, from_logits=True)
def combined_loss(pred, target):
    return 0.5 * tversky_loss(pred, target) + 0.5 * lovasz_loss(pred, target)

def compute_metrics(pred_logits, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred_logits) > threshold).long()
    tp, fp, fn, tn = smp.metrics.get_stats(pred_binary, target.long(), mode='binary')
    return {
        'iou': float(smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro')),
        'dice': float(smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro')),
        'fp_rate': float(fp.sum().item() / (fp.sum().item() + tn.sum().item() + 1e-8)),
    }

def run_epoch(model, loader, optimizer, scaler, train, threshold=0.5):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_iou, all_dice, all_fp = [], [], []
    with (torch.enable_grad() if train else torch.no_grad()):
        for imgs, masks in tqdm(loader, leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda'):
                pred = model(imgs)
                loss = combined_loss(pred, masks)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            with torch.no_grad():
                m = compute_metrics(pred, masks, threshold)
                all_iou.append(m['iou'])
                all_dice.append(m['dice'])
                all_fp.append(m['fp_rate'])
    return {'loss': total_loss / len(loader.dataset), 'iou': np.mean(all_iou), 'dice': np.mean(all_dice), 'fp_rate': np.mean(all_fp)}

In [ ]:
for p in model.encoder.parameters():
    p.requires_grad = False
print('P1: frozen | 256px | 20ep | lr=5e-4')
train_loader, val_loader, test_loader = make_loaders(train_p, val_p, test_p, 256, 16)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)
scaler = GradScaler('cuda')
best_iou = 0.0
for epoch in range(1, 21):
    tr = run_epoch(model, train_loader, optimizer, scaler, True)
    va = run_epoch(model, val_loader, optimizer, scaler, False)
    scheduler.step()
    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'model': model.state_dict()}, CKPT_DIR / 'best.pth')
    if epoch % 5 == 0:
        print(f'P1 E{epoch} iou={va["iou"]:.4f} fp={va["fp_rate"]:.3f}')
print(f'P1 done: {best_iou:.4f}')

In [ ]:
for p in model.parameters():
    p.requires_grad = True
ckpt = torch.load(CKPT_DIR / 'best.pth', weights_only=False)
model.load_state_dict(ckpt['model'])
print('P2: unfrozen | 384px | 70ep | lr=1e-4')
train_loader, val_loader, test_loader = make_loaders(train_p, val_p, test_p, 384, 16)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=35, T_mult=2, eta_min=1e-6)
scaler = GradScaler('cuda')
best_iou = 0.0
for epoch in range(1, 71):
    tr = run_epoch(model, train_loader, optimizer, scaler, True)
    va = run_epoch(model, val_loader, optimizer, scaler, False)
    scheduler.step()
    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'model': model.state_dict()}, CKPT_DIR / 'best.pth')
    if epoch % 10 == 0:
        print(f'P2 E{epoch} iou={va["iou"]:.4f} fp={va["fp_rate"]:.3f}')
print(f'P2 done: {best_iou:.4f}')

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best.pth', weights_only=False)
model.load_state_dict(ckpt['model'])
print('P3: 512px | 30ep | lr=3e-5')
train_loader, val_loader, test_loader = make_loaders(train_p, val_p, test_p, 512, 8)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)
scaler = GradScaler('cuda')
best_iou = 0.0
for epoch in range(1, 31):
    tr = run_epoch(model, train_loader, optimizer, scaler, True)
    va = run_epoch(model, val_loader, optimizer, scaler, False)
    scheduler.step()
    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'model': model.state_dict()}, CKPT_DIR / 'best.pth')
    if epoch % 10 == 0:
        print(f'P3 E{epoch} iou={va["iou"]:.4f} fp={va["fp_rate"]:.3f}')
print(f'P3 done: {best_iou:.4f}')

## Test — Threshold & Post-Processing

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best.pth', weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
train_loader, val_loader, test_loader = make_loaders(train_p, val_p, test_p, 512, 4)

print('\n=== Threshold Tuning ===')
for threshold in [0.5, 0.6, 0.7]:
    test_iou, test_fp = [], []
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            pred = model(imgs)
            m = compute_metrics(pred, masks, threshold)
            test_iou.append(m['iou'])
            test_fp.append(m['fp_rate'])
    print(f'  T={threshold}: IoU={np.mean(test_iou):.4f}  FP_Rate={np.mean(test_fp):.4f}')

In [ ]:
import shutil
DRIVE_CKPT = Path('/content/drive/MyDrive/HeritagePreservation/checkpoints/segmentor_v7_fp_fix')
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
shutil.copy2(CKPT_DIR / 'best.pth', DRIVE_CKPT / 'best.pth')
print(f'Saved to Drive: {DRIVE_CKPT / "best.pth"}')

In [ ]:
def post_process(pred_logits, threshold=0.5, min_size=50):
    pred = (torch.sigmoid(pred_logits) > threshold).cpu().numpy().astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    result = []
    for m in pred:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)
        nl, labels, stats, _ = cv2.connectedComponentsWithStats(m)
        out = np.zeros_like(m)
        for i in range(1, nl):
            if stats[i, cv2.CC_STAT_AREA] >= min_size:
                out[labels == i] = 1
        result.append(torch.from_numpy(out[None]).float())
    return torch.cat(result, 0)

print('\n=== Post-Processing (Morphological Closing + Min Size Filter) ===')
for threshold in [0.5, 0.6, 0.7]:
    test_iou, test_fp = [], []
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            pred = model(imgs)
            pred_pp = post_process(pred, threshold, min_size=50).to(DEVICE)
            m = compute_metrics(pred_pp, masks, 0.5)
            test_iou.append(m['iou'])
            test_fp.append(m['fp_rate'])
    print(f'  T={threshold}: IoU={np.mean(test_iou):.4f}  FP_Rate={np.mean(test_fp):.4f}')